# EDA: conjunto de datos Eucalyptus

> Análisis exploratorio del proyecto de clasificación multiclase. Sigue la *Guía de un proyecto de clasificación en ML* (pasos 1 y 2: split y EDA solo sobre entrenamiento). El problema, la métrica y el baseline están definidos en `00_general.ipynb`; aquí se resumen los datos y sus riesgos.

- **Datos:** *Eucalyptus Soil Conservation* (OpenML id=188, Bruce Bulloch, colección WEKA): 736 parcelas de ensayo de eucalipto en la Isla Norte de Nueva Zelanda.
- **Objetivo:** `Utility`, 5 clases ordenadas: `none` < `low` < `average` < `good` < `best`.
- **Convenciones:** idioma español; se muestra `n` junto a los porcentajes; todo el análisis usa solo el conjunto de entrenamiento.
- **Límites de los datos:** los métodos de ensayo cambiaron con el tiempo y faltan medidas en algunos sitios, sobre todo en `Mor`.

## Datos y split

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

SEMILLA = 42

In [2]:
# Eucalyptus (OpenML id=188); se guarda en caché local tras la primera descarga
datos = fetch_openml(data_id=188, as_frame=True)
X, y = datos.data, datos.target

print(f"{X.shape[0]} filas, {X.shape[1]} variables")
print(y.value_counts())

736 filas, 19 variables
Utility
good       214
none       180
average    130
low        107
best       105
Name: count, dtype: int64


In [3]:
# Split estratificado 70/15/15. El test no se toca hasta la evaluación final (paso 8).
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEMILLA
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEMILLA
)

print(f"train: {len(X_train)} | validación: {len(X_val)} | test: {len(X_test)}")

train: 515 | validación: 110 | test: 111


# 📥 Carga de datos (conjunto de entrenamiento)

> Entender qué contiene cada columna antes de cambiar nada.

---

### 🎯 Objetivo
Saber qué es una fila, cuántas hay, qué tipo tiene cada variable y qué parece sospechoso, trabajando solo con `X_train`.

### 📥 Entrada
`X_train` e `y_train` (70 % de los datos, estratificado), sin modificar. Una fila es una parcela de ensayo: un lote de semillas (`Sp`) en un sitio (`Abbrev`) y una réplica (`Rep`).

### 🛠️ Qué hacemos
1. Configurar importaciones y estilo de gráficos.
2. Revisar forma y tipos (`shape`, `info`).
3. Resumen estadístico de las variables numéricas y categóricas (`describe`).
4. Revisar los nombres de las columnas.

### 📝 Observaciones
- `X_train` tiene 515 filas y 19 variables: 14 numéricas y 5 categóricas (`Abbrev`, `Locality`, `Map_Ref`, `Latitude`, `Sp`). Los nombres de columna no tienen espacios sobrantes.
- Una fila es una parcela de ensayo (lote de semillas en un sitio y una réplica).
- Valores sospechosos para investigar primero en la exploración:
  - `DBH`: mediana 14, pero máximo 42085 (imposible para el diámetro de un árbol).
  - Valores ausentes: `Surv` 68 de 515 (13 %); `Vig`, `Ins_res`, `Stem_Fm`, `Crown_Fm` y `Brnch_Fm` 46 de 515 cada una (9 %). Hay que comprobar si son las mismas filas.
- Otros valores a vigilar: `Rep` (máximo 22, mediana 2) y `PMCno` (mínimo 1, mediana 2088).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 1. Forma y tipos

In [ ]:
print(f"X_train: {X_train.shape[0]} filas, {X_train.shape[1]} variables")
X_train.info()

## 2. Resumen de variables numéricas

In [ ]:
display(Markdown("**Variables numéricas: estadísticos**"))
X_train.describe().T

## 3. Resumen de variables categóricas

In [ ]:
display(Markdown("**Variables categóricas: número de valores distintos y más frecuente**"))
X_train.select_dtypes("category").describe().T

## 4. Nombres de columnas

In [ ]:
espacios = [c for c in X_train.columns if c != c.strip()]
print("Columnas con espacios sobrantes:", espacios or "ninguna")

# 🔍 Exploración (conjunto de entrenamiento)

> Detectar problemas con evidencia. Las decisiones se toman en el preprocesado, no aquí.

---

### 🎯 Objetivo
Encontrar valores ausentes, duplicados, valores extremos y posibles fugas de información (*data leakage*) en `X_train`.

### 📥 Entrada
`X_train` e `y_train`, sin modificar.

### 🛠️ Qué hacemos
1. Valores ausentes: cuántos hay, en qué filas y qué relación tienen con el sitio y con la clase.
2. Duplicados.
3. Valores extremos (regla del IQR y revisión de `DBH`).

### 📊 Lista de comprobación de gráficos
Cada gráfico lleva título, ejes con nombre y leyenda si hay más de una serie.

### 📝 Observaciones
- **Ausentes:** 46 filas (sitios `Mor` 41 y `Wen` 5, completos) no tienen ninguna de las seis medidas (`Surv`, `Vig`, `Ins_res`, `Stem_Fm`, `Crown_Fm`, `Brnch_Fm`). Las 46 tienen `Utility = none` (46 de 46), frente al 17 % de `none` en las filas con datos. Se interpreta como parcelas fallidas o no medidas: la ausencia señala la clase, y por eso es un riesgo de *data leakage*.
- **Ausentes en `Surv`:** otras 22 filas, todas del sitio `Cly`, solo carecen de `Surv` (clases: good 11, best 5, low 5, average 1, none 0). Se interpreta como supervivencia no registrada en ese sitio, sin relación con la clase.
- **Ausentes en `PMCno`:** 5 filas (1 %), sin revisar todavía.
- **Duplicados:** 0 filas duplicadas.
- **Extremos:** `DBH` tiene un valor de 42085 (fila 603, sitio `Paw`) frente a un máximo de 53,86 en las otras 514 filas; se considera un error de registro. Fuera de 1,5 × IQR hay 13 filas en `DBH`, 126 en `Altitude` (IQR muy estrecho, valores casi categóricos), 26 en `Brnch_Fm`, 8 en `Crown_Fm` y pocas en el resto.
- **Balance de clases:** `good` 29,1 %, `none` 24,5 %, `average` 17,7 %, `low` 14,6 %, `best` 14,2 % (desbalance moderado, sin clase dominante).
- **Asimetría:** `DBH` (22,7, por el valor 42085) y `Rep` (8,8) muy asimétricas; el resto, moderadas (|skew| < 1,1).
- **Fuga de información (*leakage*):** `Vig` es la variable más ligada a la clase (Spearman 0,79 con la escala ordenada; con `Vig` ≥ 3,5, 152 de 176 filas son `good` o `best`; con `Vig` ≤ 2, ninguna de 104). Se considera un predictor legítimo (medición de campo), a confirmar que estará disponible al predecir. Le siguen `Ins_res` y `Stem_Fm` (0,46) y `Ht` (0,45).
- **Efecto del sitio:** `Locality`, `Map_Ref`, `Latitude`, `Altitude`, `Rainfall`, `Frosts` y `Year` toman un único valor por sitio (`Abbrev`), por lo que describen el mismo sitio de varias formas. La clase depende mucho del sitio (`Mor` 41 de 41 `none`; `Wen` y `WSh` tienen solo 5 y 2 filas). Hay que tenerlo presente al validar, para que filas del mismo sitio no queden repartidas de forma engañosa entre entrenamiento y validación.
- **Redundancia entre variables:** `DBH`–`Ht` 0,82 (ambas miden tamaño) y `Stem_Fm`, `Crown_Fm`, `Brnch_Fm` entre 0,56 y 0,70.
- **Parcelas sin medidas:** al quitar las 46 filas sin medidas, `none` pasa de 126 (24,5 %) a 80 de 469 (17,1 %) y el reparto de clases queda más parejo (de 32,0 % `good` a 15,6 % `best`).
- **Para el preprocesado:** decidir cómo tratar el bloque de ausentes de `Mor`/`Wen` y `Cly`, corregir o eliminar `DBH = 42085`, reducir las variables de sitio y las redundantes (`DBH`/`Ht`, las tres de forma) y confirmar la disponibilidad de `Surv`, `Vig`, etc. al predecir.

## 1. Valores ausentes

In [ ]:
ausentes = X_train.isna().sum()
ausentes = ausentes[ausentes > 0]
tabla = pd.DataFrame({
    "ausentes": ausentes,
    "% de las filas": (ausentes / len(X_train) * 100).round(1),
})
display(Markdown(f"**Columnas con valores ausentes (n = {len(X_train)})**"))
tabla

In [ ]:
# ¿Faltan las mismas filas en todas las variables de medición?
medidas = ["Surv", "Vig", "Ins_res", "Stem_Fm", "Crown_Fm", "Brnch_Fm"]
display(Markdown("**Patrones de ausencia (True = falta) y número de filas**"))
X_train[medidas].isna().value_counts()

In [ ]:
# ¿Dónde faltan? Filas con ausencias por sitio (solo sitios afectados)
por_sitio = X_train[medidas].isna().groupby(X_train["Abbrev"], observed=True).sum()
por_sitio["n_filas"] = X_train["Abbrev"].value_counts()
display(Markdown("**Ausencias por sitio (`Abbrev`)**"))
por_sitio[por_sitio[medidas].sum(axis=1) > 0]

In [ ]:
# ¿Qué clase tienen las filas con ausencias?
falta_vig = X_train["Vig"].isna()
display(Markdown("**Clase (`Utility`) de las filas SIN `Vig` (n = {})**".format(falta_vig.sum())))
display(y_train[falta_vig].value_counts())
display(Markdown("**Clase de las filas CON `Vig` (n = {}), en %**".format((~falta_vig).sum())))
(y_train[~falta_vig].value_counts(normalize=True) * 100).round(1)

## 2. Duplicados

In [ ]:
print("Filas duplicadas en X_train:", X_train.duplicated().sum())

## 3. Valores extremos

In [ ]:
numericas = X_train.select_dtypes("number").columns
q1, q3 = X_train[numericas].quantile(0.25), X_train[numericas].quantile(0.75)
iqr = q3 - q1
fuera = ((X_train[numericas] < q1 - 1.5 * iqr) | (X_train[numericas] > q3 + 1.5 * iqr)).sum()
display(Markdown("**Filas fuera de 1,5 × IQR por variable numérica (n = {})**".format(len(X_train))))
pd.DataFrame({"fuera de IQR": fuera, "IQR": iqr}).query("`fuera de IQR` > 0")

In [ ]:
display(Markdown("**Los 5 valores más altos de `DBH`**"))
X_train.nlargest(5, "DBH")[["Abbrev", "Sp", "DBH", "Ht"]]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.boxplot(x=X_train["DBH"], ax=ax)
ax.set(title="DBH en el conjunto de entrenamiento", xlabel="DBH (diámetro a la altura del pecho)")
plt.show()

## 4. Variable objetivo: balance de clases

In [ ]:
ORDEN = ["none", "low", "average", "good", "best"]  # escala ordenada de Utility

conteo = y_train.value_counts().reindex(ORDEN)
display(Markdown(f"**Clases de `Utility` en entrenamiento (n = {len(y_train)})**"))
display(pd.DataFrame({"filas": conteo, "%": (conteo / len(y_train) * 100).round(1)}))

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(x=conteo.index, y=conteo.values, ax=ax, color="steelblue")
ax.set(title="Clases de Utility (entrenamiento)", xlabel="Utility", ylabel="Número de filas")
plt.show()

## 5. Distribuciones de las variables numéricas

In [ ]:
numericas = X_train.select_dtypes("number").columns
display(Markdown("**Asimetría (skew) por variable numérica**"))
display(X_train[numericas].skew().round(2).sort_values().to_frame("skew"))

fig, ejes = plt.subplots(4, 4, figsize=(14, 10))
for ax, col in zip(ejes.flat, numericas):
    # DBH sin el valor 42085 para que la escala sea legible (solo visualización)
    datos = X_train.loc[X_train["DBH"] < 1000, col] if col == "DBH" else X_train[col]
    sns.histplot(datos, ax=ax, bins=25)
    ax.set_title(col)
for ax in ejes.flat[len(numericas):]:
    ax.axis("off")
fig.suptitle("Distribución de las variables numéricas (entrenamiento)")
fig.tight_layout()
plt.show()

## 6. Variables categóricas: cardinalidad

In [ ]:
categoricas = X_train.select_dtypes("category").columns
resumen = pd.DataFrame({
    "valores distintos": [X_train[c].nunique() for c in categoricas],
    "filas del valor más raro": [X_train[c].value_counts().min() for c in categoricas],
    "valores con menos de 5 filas": [(X_train[c].value_counts() < 5).sum() for c in categoricas],
}, index=categoricas)
display(Markdown("**Cardinalidad de las categóricas**"))
display(resumen)

# ¿Cada sitio (Abbrev) fija las demás variables de ubicación y clima?
fijadas = ["Locality", "Map_Ref", "Latitude", "Altitude", "Rainfall", "Frosts", "Year"]
display(Markdown("**Valores distintos de cada variable dentro de un mismo sitio (máximo)**"))
X_train.groupby("Abbrev", observed=True)[fijadas].nunique().max().to_frame("máx. valores por sitio")

## 7. Relación de cada variable con la clase

In [ ]:
# Tabla de clases por sitio (el sitio condiciona mucho la clase)
tabla_sitio = pd.crosstab(X_train["Abbrev"], y_train)[ORDEN]
tabla_sitio["n"] = tabla_sitio.sum(axis=1)
display(Markdown("**Filas por sitio y clase**"))
display(tabla_sitio.sort_values("n"))

In [ ]:
# Medidas por clase (cada caja usa solo las filas con dato; se indica n por clase)
medidas_clase = ["Vig", "Ht", "DBH", "Surv", "Ins_res", "Stem_Fm"]
fig, ejes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(ejes.flat, medidas_clase):
    datos = X_train.assign(Utility=y_train)
    if col == "DBH":
        datos = datos[datos["DBH"] < 1000]  # solo visualización, sin el valor 42085
    sns.boxplot(data=datos, x="Utility", y=col, order=ORDEN, ax=ax)
    ax.set_title(col)
fig.suptitle("Variables por clase de Utility (entrenamiento)")
fig.tight_layout()
plt.show()

display(Markdown("**n con dato por clase**"))
X_train[medidas_clase].notna().groupby(y_train).sum().loc[ORDEN]

## 8. Correlaciones

In [ ]:
# Correlación de Spearman de cada variable numérica con la clase ordenada (none=0 ... best=4)
y_ordinal = y_train.astype(str).map({c: i for i, c in enumerate(ORDEN)})
corr_objetivo = X_train[numericas].corrwith(y_ordinal, method="spearman").round(2).sort_values()
display(Markdown("**Spearman con la clase ordenada (por pares con dato)**"))
display(corr_objetivo.to_frame("rho"))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(X_train[numericas].corr(method="spearman"), annot=True, fmt=".1f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación de Spearman entre variables numéricas (entrenamiento)")
plt.show()

## 9. Efecto de las parcelas sin medidas en el reparto de clases

In [ ]:
sin_medidas = X_train["Vig"].isna()  # son las mismas 46 filas que no tienen ninguna de las seis medidas
conteo_todas = y_train.value_counts().reindex(ORDEN)
conteo_con = y_train[~sin_medidas].value_counts().reindex(ORDEN)
comparacion = pd.DataFrame({
    f"con todas (n = {len(y_train)})": conteo_todas,
    "%": (conteo_todas / len(y_train) * 100).round(1),
    f"sin las parcelas sin medidas (n = {(~sin_medidas).sum()})": conteo_con,
    "% ": (conteo_con / (~sin_medidas).sum() * 100).round(1),
})
display(Markdown("**Reparto de clases con y sin las parcelas sin medidas (entrenamiento)**"))
comparacion

# 📝 Conclusión del EDA y traspaso al equipo

> Aquí termina el EDA común. El preprocesado lo hace cada integrante en su propio trabajo, antes de entrenar su modelo; después el equipo compara los modelos y elige uno.

---

### ✅ Qué hemos comprobado (conjunto de entrenamiento, n = 515)
- **Tarea:** clasificación de `Utility` en 5 clases ordenadas (`none` < `low` < `average` < `good` < `best`), con desbalance moderado (de 29,1 % a 14,2 %).
- **Calidad:** sin duplicados; un error evidente en `DBH` (42085); ausentes concentrados en dos bloques (`Mor`/`Wen` sin ninguna medida, `Cly` sin `Surv`) y 5 en `PMCno`.
- **Señal:** `Vig` es la variable más ligada a la clase (Spearman 0,79), seguida de `Ins_res`, `Stem_Fm` y `Ht` (≈ 0,45).

### ⚠️ Riesgos a tener presentes
- **Las 46 filas sin medidas son todas `none`** (46 de 46): la ausencia delata la clase. Un modelo que use los ausentes (o un imputador que los rellene) puede parecer mejor de lo que es y fallar con datos nuevos que sí traigan medidas.
- **El sitio condiciona la clase** y siete columnas describen lo mismo (el sitio). Repartir filas del mismo sitio entre entrenamiento y validación puede inflar las métricas.
- **`Vig` y otras medidas** deben estar disponibles al predecir; hay que confirmarlo.
- **Límites:** 736 filas en total, grupos pequeños (p. ej. `WSh` 2 filas, `Wen` 5), y los métodos de ensayo cambiaron con el tiempo.

### 🔧 Decisiones que quedan para el preprocesado de cada integrante
Cada persona decide y justifica (`⚖️ Decisión / Por qué`) cómo trata:
1. Los ausentes de `Cly` (`Surv`) y los de `PMCno`. (Las parcelas sin medidas de `Mor`/`Wen` tienen una propuesta de regla común, más abajo.)
2. `DBH = 42085` (corregir, eliminar o limitar).
3. Las variables de sitio redundantes y las parejas `DBH`–`Ht` y `Stem_Fm`/`Crown_Fm`/`Brnch_Fm`.
4. La codificación de categóricas (`Sp` tiene 27 valores y `Abbrev` 16, con valores raros) y el escalado según el modelo.

### 🧹 Propuesta de regla común: quitar las parcelas sin medidas
Se propone **quitar del modelado las parcelas sin ninguna medida** (las 46 de `Mor`/`Wen` en entrenamiento, todas `none`), aplicando **la misma regla a entrenamiento, validación y test**:
- *Por qué:* la ausencia delata la clase (46 de 46), así que un modelo que la use parece mejor de lo que será, y esas filas son el 37 % de la clase `none` en entrenamiento (46 de 126).
- *Alcance:* el modelo solo vale para parcelas con medidas; una parcela sin medidas se trataría fuera del modelo con una regla simple, y habría que decirlo en el informe y en la aplicación.
- *Qué no cambia:* la relación del vigor con la clase, porque esas filas no tenían vigor.
- *Decisión de equipo:* como cada integrante preprocesa por su cuenta, debe acordarse entre todos (si no, los modelos no se pueden comparar). Alternativa: que todos informen las métricas con y sin esas parcelas.

### 🤝 Para poder comparar los modelos con justicia
Hay que acordar entre todos, antes de entrenar: el mismo reparto train/validación/test (`SEMILLA = 42`, ya hecho arriba), el mismo esquema de validación cruzada (`StratifiedKFold`, 5 folds) y las mismas métricas (F1 macro, accuracy y F1 por clase). El test no se toca hasta elegir el modelo final.